In [1]:
import sys
import os

sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np

from src.preprocessing import clean_data, create_target

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
df = pd.read_csv('../data/raw/geometry_data.csv')

df.shape

(9997, 49)

In [3]:
df = clean_data(df)

df = create_target(df)

df.shape

(9877, 50)

In [4]:
drop_cols = [
    'file_id',
    'quality_class',
    'num_self_intersections',
    'num_boundary_edges',
    'num_duplicated_faces',
    'num_geometrical_degenerated_faces'
]

X = df.drop(columns=drop_cols)
y = df['quality_class']

print(X.shape)
print(y.shape)

(9877, 44)
(9877,)


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [6]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [7]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)

lr_model.fit(X_train_scaled, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

In [8]:
from sklearn.metrics import classification_report

y_pred_lr = lr_model.predict(X_test_scaled)

print(classification_report(y_test, y_pred_lr))

              precision    recall  f1-score   support

           0       1.00      0.98      0.99      1050
           1       0.79      0.81      0.80       480
           2       0.64      0.49      0.56       333
           3       0.44      0.67      0.53        99
           4       0.21      0.71      0.33        14

    accuracy                           0.84      1976
   macro avg       0.62      0.73      0.64      1976
weighted avg       0.85      0.84      0.84      1976



In [9]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_neighbors=5)

knn_model.fit(X_train_scaled, y_train)

KNeighborsClassifier()

In [10]:
y_pred_knn = knn_model.predict(X_test_scaled)

print(classification_report(y_test, y_pred_knn))

              precision    recall  f1-score   support

           0       0.97      0.98      0.98      1050
           1       0.78      0.81      0.80       480
           2       0.64      0.62      0.63       333
           3       0.54      0.43      0.48        99
           4       0.36      0.29      0.32        14

    accuracy                           0.85      1976
   macro avg       0.66      0.63      0.64      1976
weighted avg       0.84      0.85      0.85      1976



In [12]:
from sklearn.metrics import f1_score

k_values = [3, 5, 7]

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    
    knn.fit(X_train_scaled, y_train)
    
    y_pred = knn.predict(X_test_scaled)
    
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    
    print(f'k={k}')
    print(f'Macro F1-score: {macro_f1:.4f}')
    print(classification_report(y_test, y_pred))
    print('-' * 50)

k=3
Macro F1-score: 0.6849
              precision    recall  f1-score   support

           0       0.97      0.98      0.98      1050
           1       0.77      0.82      0.79       480
           2       0.67      0.62      0.64       333
           3       0.60      0.51      0.55        99
           4       0.50      0.43      0.46        14

    accuracy                           0.85      1976
   macro avg       0.70      0.67      0.68      1976
weighted avg       0.85      0.85      0.85      1976

--------------------------------------------------
k=5
Macro F1-score: 0.6409
              precision    recall  f1-score   support

           0       0.97      0.98      0.98      1050
           1       0.78      0.81      0.80       480
           2       0.64      0.62      0.63       333
           3       0.54      0.43      0.48        99
           4       0.36      0.29      0.32        14

    accuracy                           0.85      1976
   macro avg       0.66   

In [13]:
from sklearn.tree import DecisionTreeClassifier

depths = [5, 10, None]

for depth in depths:
    
    tree = DecisionTreeClassifier(
        max_depth=depth,
        random_state=42,
        class_weight='balanced'
    )
    
    tree.fit(X_train, y_train)
    
    y_pred_tree = tree.predict(X_test)
    
    macro_f1 = f1_score(y_test, y_pred_tree, average='macro')
    
    print(f'max_depth={depth}')
    print(f'Macro F1-score: {macro_f1:.4f}')
    print(classification_report(y_test, y_pred_tree))
    print('-' * 50)

max_depth=5
Macro F1-score: 0.7439
              precision    recall  f1-score   support

           0       1.00      0.98      0.99      1050
           1       0.96      0.82      0.89       480
           2       0.78      0.83      0.81       333
           3       0.52      0.72      0.60        99
           4       0.28      0.93      0.43        14

    accuracy                           0.90      1976
   macro avg       0.71      0.86      0.74      1976
weighted avg       0.92      0.90      0.91      1976

--------------------------------------------------
max_depth=10
Macro F1-score: 0.7910
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1050
           1       0.93      0.91      0.92       480
           2       0.86      0.84      0.85       333
           3       0.73      0.75      0.74        99
           4       0.32      0.79      0.46        14

    accuracy                           0.93      1976
   macro 

In [14]:
from sklearn.ensemble import RandomForestClassifier

rf_configs = [
    (100, None),
    (100, 10),
    (200, 10),
    (200, 20)
]

for n_estimators, depth in rf_configs:
    
    rf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=depth,
        random_state=42,
        class_weight='balanced'
    )
    
    rf.fit(X_train, y_train)
    
    y_pred_rf = rf.predict(X_test)
    
    macro_f1 = f1_score(y_test, y_pred_rf, average='macro')
    
    print(f'n_estimators={n_estimators}, max_depth={depth}')
    print(f'Macro F1-score: {macro_f1:.4f}')
    print(classification_report(y_test, y_pred_rf))
    print('-' * 60)

n_estimators=100, max_depth=None
Macro F1-score: 0.7484
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1050
           1       0.93      0.93      0.93       480
           2       0.84      0.86      0.85       333
           3       0.73      0.77      0.75        99
           4       0.50      0.14      0.22        14

    accuracy                           0.94      1976
   macro avg       0.80      0.74      0.75      1976
weighted avg       0.94      0.94      0.94      1976

------------------------------------------------------------
n_estimators=100, max_depth=10
Macro F1-score: 0.7902
              precision    recall  f1-score   support

           0       1.00      0.98      0.99      1050
           1       0.94      0.90      0.92       480
           2       0.83      0.85      0.84       333
           3       0.67      0.85      0.75        99
           4       0.46      0.43      0.44        14

    accuracy  

In [15]:
from sklearn.ensemble import GradientBoostingClassifier

gb_configs = [
    (100, 0.1),
    (200, 0.05),
    (200, 0.1)
]

for n_estimators, lr in gb_configs:
    
    gb = GradientBoostingClassifier(
        n_estimators=n_estimators,
        learning_rate=lr,
        random_state=42
    )
    
    gb.fit(X_train, y_train)
    
    y_pred_gb = gb.predict(X_test)
    
    macro_f1 = f1_score(y_test, y_pred_gb, average='macro')
    
    print(f'n_estimators={n_estimators}, learning_rate={lr}')
    print(f'Macro F1-score: {macro_f1:.4f}')
    print(classification_report(y_test, y_pred_gb))
    print('-' * 60)

n_estimators=100, learning_rate=0.1
Macro F1-score: 0.7798
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1050
           1       0.91      0.92      0.91       480
           2       0.86      0.82      0.84       333
           3       0.73      0.82      0.77        99
           4       0.42      0.36      0.38        14

    accuracy                           0.93      1976
   macro avg       0.78      0.78      0.78      1976
weighted avg       0.93      0.93      0.93      1976

------------------------------------------------------------
n_estimators=200, learning_rate=0.05
Macro F1-score: 0.7947
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1050
           1       0.91      0.92      0.91       480
           2       0.86      0.83      0.85       333
           3       0.75      0.83      0.79        99
           4       0.56      0.36      0.43        14

    a

In [16]:
results = pd.DataFrame({
    'Model': [
        'Logistic Regression',
        'KNN (k=3)',
        'Decision Tree',
        'Random Forest',
        'Gradient Boosting'
    ],
    
    'Accuracy': [
        0.84,
        0.85,
        0.92,
        0.94,
        0.94
    ],
    
    'Macro F1-score': [
        0.64,
        0.68,
        0.79,
        0.79,
        0.79
    ]
})

results

,Model,Accuracy,Macro F1-score
0,Logistic Regression,0.84,0.64
1,KNN (k=3),0.85,0.68
2,Decision Tree,0.92,0.79
3,Random Forest,0.94,0.79
4,Gradient Boosting,0.94,0.79


В рамках CP2 были протестировны несколько моделей машинного обучения: Logistic Regression, K-Nearest Neihbors, Decision Tree, Random Forest, Gradient Boosting;
В качестве основной метрики использовался macro F1-score. Accuracy в данном случае недостаточно показательна, поскольку большая часть объектов относится к классу 0.
Logistic Regression использовалась как baseline-модель и показала наименьшее качество среди протестированных алгоритмов. 
KNN улучшил качество классификации, особенно для редких классов, при использовании небольшого количества соседей.
Decision Tree значительно повысила качество модели благодаря способности учитывать нелинейные зависимости между признаками.
Методы Random Forest и Gradient Boosting показали лучшие результаты по accuracy и macro F1-score. Также Random Forest и Gradient Boosting продемонстрировали наиболее стабильное качество и лучше справлялись с дисбалансом классов.
В качестве финальной модели для дальнейшей работы может быть выбран Random Forest, так как он показывает высокое качество классификации и хорошо работает с табличными данными и нелинейными зависимостями.

Далее для дополнительного эксперимента будет применён метод PCA для уменьшения размерности признакового пространства при сохранении большей части информации.

In [17]:
from sklearn.decomposition import PCA

In [18]:
pca = PCA(n_components=0.95)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(X_train_pca.shape)
print(X_test_pca.shape)

(7901, 27)
(1976, 27)


In [19]:
lr_pca = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)

lr_pca.fit(X_train_pca, y_train)

y_pred_lr_pca = lr_pca.predict(X_test_pca)

print(classification_report(y_test, y_pred_lr_pca))

macro_f1_lr_pca = f1_score(
    y_test,
    y_pred_lr_pca,
    average='macro'
)

print(f'Macro F1-score: {macro_f1_lr_pca:.4f}')

              precision    recall  f1-score   support

           0       1.00      0.98      0.99      1050
           1       0.77      0.80      0.79       480
           2       0.64      0.46      0.53       333
           3       0.39      0.61      0.48        99
           4       0.20      0.79      0.32        14

    accuracy                           0.83      1976
   macro avg       0.60      0.73      0.62      1976
weighted avg       0.85      0.83      0.83      1976

Macro F1-score: 0.6210


In [20]:
rf_pca = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced'
)

rf_pca.fit(X_train_pca, y_train)

y_pred_rf_pca = rf_pca.predict(X_test_pca)

print(classification_report(y_test, y_pred_rf_pca))

macro_f1_rf_pca = f1_score(
    y_test,
    y_pred_rf_pca,
    average='macro'
)

print(f'Macro F1-score: {macro_f1_rf_pca:.4f}')

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1050
           1       0.82      0.86      0.84       480
           2       0.70      0.69      0.70       333
           3       0.63      0.53      0.57        99
           4       0.38      0.21      0.27        14

    accuracy                           0.88      1976
   macro avg       0.70      0.66      0.67      1976
weighted avg       0.87      0.88      0.88      1976

Macro F1-score: 0.6731


Применение метода не улучшило качество моделей. Для LR снижение было небольшим (по сравнению с исходными признаками), а для RF оно оказалось более заметным: accuracy уменьшилась с 0,94 до 0,88, а macro F1-scoreс 0,79 до 0,67.